# MSTR - Robotics: Schema Monitor

### Load libraries

In [ ]:
import yaml
from mstr_robotics._paths import REPO_ROOT, CONFIG_DIR, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
from mstr_robotics.read_out_prj_obj import ReadSchema,ReadGen

from mstr_robotics.report import Cube,Rep
from mstr_robotics.mstr_pandas import DfHelper
from mstr_robotics.mstr_classes import  get_conn

import pandas as pd
import json

i_rep=Rep()
i_cube=Cube()
i_read_gen=ReadGen()
i_df_helper=DfHelper()
i_read_schema=ReadSchema()
run_prop_d={}


#in my environment the PA project is on the same maschine
#open parameter files
try:
    with open(USER_CONFIG, 'r') as openfile:
        user_d = yaml.safe_load(openfile)
    conn_params = user_d["conn_params"]
except Exception as err:
    print(err)
    
try:   
    with open(CONFIG_DIR / "jupyter_objects_d.yml", "r") as openfile:
        jupyter_objects_d = yaml.safe_load(openfile)
    nb_d = jupyter_objects_d["jup_schema_monitor"]
except Exception as err:
    print(err)


### Config

In [2]:
#Schema information
#object ids are maintained in ..\config\jupyter_objects_d.yml
#set a cube id to None to create a new cube on upload

project_id = conn_params["project_id"]
pa_project_id = user_d["mstr_projects"]["pa_project_id"]
pa_base_url = conn_params["base_url"]


cube_folder_id=nb_d["folders"]["cube_folder_id"] #Schema Monitor

cbe_tbl_att_fct_mapping_name="tbl_att_fct_mapping"
cbe_tbl_att_fct_mapping_id=nb_d["cubes"]["cbe_tbl_att_fct_mapping_id"]

cbe_att_form_exp_name="att_form_exp"
cbe_att_form_exp_id=nb_d["cubes"]["cbe_att_form_exp_id"]

cbe_fact_exp_name="fact_exp"
cbe_fact_exp_id=nb_d["cubes"]["cbe_fact_exp_id"]

cbe_table_exp_name="table_exp"
cbe_table_exp_id=nb_d["cubes"]["cbe_table_exp_id"]

# blended object and PA/EM information
cbe_obj_depn_rep_col_usage_name="obj_depn_rep_col_usage"
cbe_obj_depn_rep_col_usage_id=None

## run readout


In [3]:
# connect to MD & Platform Analytics
conn=get_conn(**conn_params)
conn.select_project(project_id)
conn.headers['Content-type'] = "application/json"
           
pa_conn=get_conn(**conn_params)
pa_conn.select_project(pa_project_id)


Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'MicroStrategy Tutorial' with ID: 'B7CA92F04B9FAE8D941C3E9B7E0CD754'
Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'MicroStrategy Tutorial' with ID: 'B7CA92F04B9FAE8D941C3E9B7E0CD754'
Project selected in Connection object:
Project object named: 'Platform Analytics' with ID: '7576CD5F48607C21C914ACBE053B259B'


### read out the schema 

In [4]:
schema_mappings_d=i_read_schema.table_mappings(conn=conn,run_prop_d=run_prop_d)

#better readabillity in the next cell
att_form_exp_df=schema_mappings_d["att_form_exp_df"]
fact_exp_df=schema_mappings_d["fact_exp_df"]
table_df=schema_mappings_d["table_df"]

#prepare data for cubing
att_form_exp_d_l=[{"df":att_form_exp_df,"tbl_name":"att_form_exp_df", "update_policy":"Replace"}]
fact_exp_d_l=[{"df":fact_exp_df,"tbl_name":"fact_exp_df", "update_policy":"Replace"}]
table_d_l=[{"df":table_df,"tbl_name":"table_df", "update_policy":"Replace"}]

In [5]:
# Map schema objects & logical tables
#here we build a primary key data set by joinning dataframes

tbl_att_fct_df = pd.merge(table_df[["project_id","table_id","table_name", "column_id","column_name","physicalTable_id","physicalTable_name"]],
                          att_form_exp_df[["project_id", 'table_id', "column_id", "attribute_id","attribute_name", "form_expressionId","form_name"]],
                          on=['project_id', 'table_id', 'column_id'], how='left')
i_df_helper.clean_double_col(df=tbl_att_fct_df)
tbl_att_fct_df = pd.merge(tbl_att_fct_df, 
                          fact_exp_df[["project_id", "column_id","fact_id","fact_name","fact_expressionId"]],
                          on=['project_id', 'column_id'], how='left')
i_df_helper.clean_double_col(df=tbl_att_fct_df)
tbl_att_fct_df["phys_tbl_col_id"]=tbl_att_fct_df["physicalTable_id"]+ "_"+ tbl_att_fct_df["column_id"]
tbl_att_fct_d_l=[{"df":tbl_att_fct_df,"tbl_name":"tbl_att_fct_df", "update_policy":"Replace"}]

### Read out Platform Analytics

In [6]:
# filter the relevant project using an
# element prompt in PA

pa_project_element_id=nb_d["misc"]["pa_project_element_id"] # here you see the element_id of MicroStrategy Tutorial in PA
pa_attribute_project_id=nb_d["prompts"]["pa_attribute_project_id"]
pa_element_prompt_id=nb_d["prompts"]["pa_element_prompt_id"]
pa_conn.select_project(pa_project_id)
prompt_answ=f'{{"prompts": [{{"id": "{pa_element_prompt_id}", "type": "ELEMENTS","answers":[{{"id": "h{pa_project_element_id};{pa_attribute_project_id}"}}]}} ]}}'

print("Element Prompt Answer= " + prompt_answ)

Element Prompt Answer= {"prompts": [{"id": "EF66D49D40D0E799E1D1909A65085E97", "type": "ELEMENTS","answers":[{"id": "h7397622336377065472;3A0BE6C741DE7CDD4A4C01925B5A04E9"}]} ]}


In [7]:
# read out the execution times 
# of reports per month

report_usage=nb_d["reports"]["report_usage"]


instance_id=i_rep.open_Instance(conn=pa_conn,report_id=report_usage)
prp_resp=i_rep.set_inst_prompt_ans(conn=pa_conn,report_id=report_usage,instance_id=instance_id, prompt_answ=prompt_answ)
report_usage_df=i_rep.rep_to_dataframe(conn=pa_conn,report_id=report_usage,instance_id=instance_id)

rep_usage_d_l=[{"df":report_usage_df,"tbl_name":"rep_usage_df", "update_policy":"Replace"}]

In [8]:
# read out the execution times 
# of views and columns per report

views_cols_usage=nb_d["reports"]["views_cols_usage"]
#get data
instance_id=i_rep.open_Instance(conn=pa_conn,report_id=views_cols_usage)
prp_resp=i_rep.set_inst_prompt_ans(conn=pa_conn,report_id=report_usage,instance_id=instance_id, prompt_answ=prompt_answ)
view_col_usage_df=i_rep.rep_to_dataframe(conn=pa_conn,report_id=views_cols_usage,instance_id=instance_id)

#prepare cube upload
view_col_usage_d_l=[{"df":view_col_usage_df,"tbl_name":"view_col_usage_df", "update_policy":"Replace"}]

### Read out report definitions

In [9]:

#count_only_fg=True counts dependent objects
#count_only_fg=False list dependent objects

#info_level = "base" provides the folder id
#info_level = "base_path" provides the full folder path 

obj_type_l=["3"] #only check reports
obj_depn_df =i_read_gen.chk_by_obj_type(conn=conn #,project_id=project_id
                          ,obj_type_l=obj_type_l, info_level = "base_path",mtdi_id=None
                          ,count_only_fg=True,run_prop_d=run_prop_d)

#adjusting the name here, simplyfies the mapping of the attributes in the dossier
obj_depn_df.rename(columns={'id': 'report_id'}, inplace=True)

#prepare cube upload
obj_depn_d_l=[{"df":obj_depn_df,"tbl_name":"obj_depn_df", "update_policy":"Replace"}]

### Cubing

In [10]:
# if mtdi_id = None a new cube is created
# if mtdi_id <> None an existing cube is update
# if the mtdi_id does not exists or you provide different data structures
# you'll get an error msg


#primery key cube schema usage
i_cube.upload_cube_mult_table(conn=conn,tbl_upd_dict=tbl_att_fct_d_l,mtdi_id=cbe_tbl_att_fct_mapping_id , cube_name=cbe_tbl_att_fct_mapping_name,folder_id=cube_folder_id,force=True)
#attribute form expressions 
i_cube.upload_cube_mult_table(conn=conn,tbl_upd_dict=att_form_exp_d_l,mtdi_id=cbe_att_form_exp_id , cube_name=cbe_att_form_exp_name,folder_id=cube_folder_id,force=True)
#fact expressions
i_cube.upload_cube_mult_table(conn=conn,tbl_upd_dict=fact_exp_d_l,mtdi_id=cbe_fact_exp_id , cube_name=cbe_fact_exp_name,folder_id=cube_folder_id,force=True)
#table definitions
i_cube.upload_cube_mult_table(conn=conn,tbl_upd_dict=table_d_l,mtdi_id=cbe_table_exp_id , cube_name=cbe_table_exp_name,folder_id=cube_folder_id,force=True)

#define multi table cube
#ensure that derived attributes are mapped 
#over common column names
obj_depn_rep_col_usage_l=[]
obj_depn_rep_col_usage_l.extend(obj_depn_d_l)
obj_depn_rep_col_usage_l.extend(rep_usage_d_l)
obj_depn_rep_col_usage_l.extend(view_col_usage_d_l)
i_cube.upload_cube_mult_table(conn=conn,tbl_upd_dict=obj_depn_rep_col_usage_l, mtdi_id=cbe_obj_depn_rep_col_usage_id,cube_name=cbe_obj_depn_rep_col_usage_name ,folder_id=cube_folder_id,force=True)
conn.close()
pa_conn.close()


SuperCube object named: 'tbl_att_fct_map' with ID: '3673EC9146EE4DF3517725BBA97A4B36'


  0%|          | 0/1 [00:00<?, ?it/s]

Super cube 'tbl_att_fct_map' published successfully.
SuperCube object named: 'att_form_exp' with ID: '3E312E64489040609C0A63800A4E3BAE'


  0%|          | 0/1 [00:00<?, ?it/s]

Super cube 'att_form_exp' published successfully.
SuperCube object named: 'fact_exp' with ID: '5D8A265D4A08641AAB76DAB74440A41C'


...

Super cube 'table_exp' published successfully.
Created super cube 'obj_depn_rep_col_usage' with ID: '9849389D4FDD2C8CCC38588FC7939308'.


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Super cube 'obj_depn_rep_col_usage' published successfully.
Connection to Strategy One Intelligence Server has been closed.
Connection to Strategy One Intelligence Server has been closed.
